# BTQT2 – Tiền xử lý Telco Customer Churn bằng dbt & BigQuery (Bronze → Silver)

**Notebook:** `telco_bronze_to_silver.ipynb` · **GCP Project:** `btqt2-telco-dbt-bigquery`
**Bronze:** `bronze_telco.customers_raw` · **Silver:** `silver_telco.customers_cleaned` · **dbt project:** `dbt_telco`

| Phần | Nội dung | Phụ trách |
|---|---|---|
| 0 | Thiết lập môi trường & xác thực Google Cloud | Kha |
| 1 | Tạo dataset Bronze / Silver | Kha |
| 2 | Nạp CSV vào Bronze (`customers_raw`) | Kha |
| 3 | Kiểm tra sau nạp (CSV ↔ Bronze) | Kha |
| 4 | Cài dbt, cấu hình `profiles.yml`, `dbt debug` | Kha (môi trường) |
| 5 | Model dbt `stg_telco_customers`, `customers_cleaned`, `dbt run` | Duy Lê |
| 6 | `schema.yml`, `dbt test`, so sánh Bronze ↔ Silver | Khôi |

### Hướng dẫn chạy
1. Mở notebook trên Google Colab, đăng nhập đúng tài khoản Google đã được cấp quyền vào project.
2. Chạy lần lượt từ trên xuống (**Runtime → Run all** hoặc `Ctrl+F9`).
3. Ô **0.2** sẽ hiện popup đăng nhập Google → chọn tài khoản → **Allow**.
4. Ô **2.1** sẽ hiện nút **Choose Files** → chọn file `telco_customer_churn.csv`.
5. Mỗi ô có thể chạy lại nhiều lần: dataset tạo bằng `exists_ok=True`, bảng Bronze nạp bằng `WRITE_TRUNCATE` (ghi đè, không bị nhân đôi dữ liệu).

## 0. Thiết lập môi trường
### 0.1. Cấu hình chung


In [5]:
PROJECT_ID     = "avid-booster-511409-t2"   # Project ID trên Google Cloud
LOCATION       = "asia-southeast1"            # Singapore – Bronze, Silver và dbt PHẢI cùng location
BRONZE_DATASET = "bronze_telco"
SILVER_DATASET = "silver_telco"
BRONZE_TABLE   = "customers_raw"
SILVER_TABLE   = "customers_cleaned"
CSV_FILE       = "telco_customer_churn.csv"   # tên chuẩn (gốc: WA_Fn-UseC_-Telco-Customer-Churn.csv)

BRONZE_TABLE_ID = f"{PROJECT_ID}.{BRONZE_DATASET}.{BRONZE_TABLE}"
print("Bronze table:", BRONZE_TABLE_ID)

Bronze table: avid-booster-511409-t2.bronze_telco.customers_raw


### 0.2. Xác thực Google Cloud
Popup đăng nhập sẽ hiện ra → chọn tài khoản → **Allow**.

In [6]:
from google.colab import auth
auth.authenticate_user()

import os
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
!gcloud config set project {PROJECT_ID}
print("Đã xác thực Google Cloud")

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Đã xác thực Google Cloud


### 0.3. Khởi tạo BigQuery client và kiểm tra kết nối

In [7]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print("Kết nối project:", client.project)
print("Thử truy vấn:", list(client.query("SELECT 1 AS ok").result())[0].ok)

Kết nối project: avid-booster-511409-t2
Thử truy vấn: 1


## 1. Tạo dataset Bronze và Silver

In [8]:
def create_dataset(name, description):
    ds = bigquery.Dataset(f"{PROJECT_ID}.{name}")
    ds.location = LOCATION
    ds.description = description
    ds = client.create_dataset(ds, exists_ok=True)
    print(f"{ds.full_dataset_id}  |  location = {ds.location}")

create_dataset(BRONZE_DATASET, "Bronze layer – dữ liệu Telco Customer Churn gốc, giữ nguyên dạng chuỗi")
create_dataset(SILVER_DATASET, "Silver layer – dữ liệu đã làm sạch bằng dbt (dbt_telco)")

print("\nCác dataset trong project:")
for d in client.list_datasets():
    print(" -", d.dataset_id)

avid-booster-511409-t2:bronze_telco  |  location = asia-southeast1
avid-booster-511409-t2:silver_telco  |  location = asia-southeast1

Các dataset trong project:
 - bronze_telco
 - silver_telco


## 2. Nạp dữ liệu vào Bronze
### 2.1. Tải file CSV lên Colab
Nếu file đã có sẵn trong `/content` thì ô này bỏ qua bước upload. File upload có tên khác (ví dụ tên gốc từ Kaggle) sẽ được đổi về tên chuẩn.

In [9]:
import os, shutil
if not os.path.exists(CSV_FILE):
    from google.colab import files
    uploaded = files.upload()
    src = list(uploaded.keys())[0]
    if src != CSV_FILE:
        shutil.move(src, CSV_FILE)
print("File:", CSV_FILE, "|", os.path.getsize(CSV_FILE), "bytes")

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv
File: telco_customer_churn.csv | 977501 bytes


### 2.2. Đọc CSV ở dạng thô
Nguyên tắc Bronze: **giữ nguyên dữ liệu gốc**. Vì vậy:
- `dtype=str`: mọi cột đọc thành chuỗi, pandas không tự ép kiểu.
- `keep_default_na=False`: không biến chuỗi rỗng / khoảng trắng thành NaN → 11 giá trị `' '` của `TotalCharges` được giữ nguyên để dbt xử lý.

In [10]:
df_raw = pd.read_csv(CSV_FILE, dtype=str, keep_default_na=False)
print("Kích thước:", df_raw.shape)          # kỳ vọng (7043, 21)
assert df_raw.shape == (7043, 21), "Số dòng/cột không khớp dataset gốc"
print("TotalCharges chỉ chứa khoảng trắng:", (df_raw["TotalCharges"].str.strip() == "").sum())  # kỳ vọng 11
df_raw.head()

Kích thước: (7043, 21)
TotalCharges chỉ chứa khoảng trắng: 11


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


### 2.3. Thêm cột truy vết (metadata)
- `_source_file`: tên file nguồn
- `_loaded_at`: thời điểm nạp (UTC)

Hai cột bắt đầu bằng `_` để phân biệt với 21 cột nghiệp vụ.

In [11]:
from datetime import datetime, timezone

df_bronze = df_raw.copy()
df_bronze["_source_file"] = CSV_FILE
df_bronze["_loaded_at"]   = pd.Timestamp(datetime.now(timezone.utc))
print(df_bronze.dtypes)

customerID                       object
gender                           object
SeniorCitizen                    object
Partner                          object
Dependents                       object
tenure                           object
PhoneService                     object
MultipleLines                    object
InternetService                  object
OnlineSecurity                   object
OnlineBackup                     object
DeviceProtection                 object
TechSupport                      object
StreamingTV                      object
StreamingMovies                  object
Contract                         object
PaperlessBilling                 object
PaymentMethod                    object
MonthlyCharges                   object
TotalCharges                     object
Churn                            object
_source_file                     object
_loaded_at          datetime64[us, UTC]
dtype: object


### 2.4. Upload lên BigQuery `bronze_telco.customers_raw`
Schema khai báo tường minh: 21 cột STRING (giữ tên cột gốc) + 2 cột metadata.

In [12]:
schema = [bigquery.SchemaField(c, "STRING", mode="NULLABLE") for c in df_raw.columns]
schema += [
    bigquery.SchemaField("_source_file", "STRING",    description="Tên file CSV nguồn"),
    bigquery.SchemaField("_loaded_at",   "TIMESTAMP", description="Thời điểm nạp vào Bronze (UTC)"),
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_TRUNCATE",   # chạy lại sẽ ghi đè, không nhân đôi dữ liệu
)
job = client.load_table_from_dataframe(df_bronze, BRONZE_TABLE_ID, job_config=job_config)
job.result()

table = client.get_table(BRONZE_TABLE_ID)
table.description = "Bronze – Telco Customer Churn gốc (Kaggle/IBM), 21 cột STRING + metadata nạp"
client.update_table(table, ["description"])
print(f"Đã nạp {table.num_rows} dòng, {len(table.schema)} cột vào {BRONZE_TABLE_ID}")

Đã nạp 7043 dòng, 23 cột vào avid-booster-511409-t2.bronze_telco.customers_raw


### 2.5. Xem schema và dữ liệu mẫu trên Bronze


In [13]:
pd.DataFrame([(f.name, f.field_type, f.mode) for f in table.schema],
             columns=["column", "type", "mode"])

,column,type,mode
0,customerID,STRING,NULLABLE
1,gender,STRING,NULLABLE
2,SeniorCitizen,STRING,NULLABLE
3,Partner,STRING,NULLABLE
4,Dependents,STRING,NULLABLE
5,tenure,STRING,NULLABLE
6,PhoneService,STRING,NULLABLE
7,MultipleLines,STRING,NULLABLE
8,InternetService,STRING,NULLABLE
9,OnlineSecurity,STRING,NULLABLE


In [14]:
client.query(f"SELECT * FROM `{BRONZE_TABLE_ID}` LIMIT 10").to_dataframe()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,_source_file,_loaded_at
0,3473-XIIIT,Female,0,Yes,No,16,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100,1534.75,Yes,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
1,8709-KRDVL,Female,0,No,No,33,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100,3320.6,No,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
2,8397-MVTAZ,Male,0,Yes,No,34,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3480,Yes,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
3,1666-JXLKU,Female,0,No,No,37,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3810.55,No,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
4,2669-OIDSD,Female,0,Yes,No,48,Yes,No,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.05,4834,No,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
5,2632-UCGVD,Male,1,Yes,No,66,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.05,6871.9,Yes,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
6,9647-ERGBE,Female,0,Yes,Yes,30,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3046.15,Yes,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
7,8634-MPHTR,Male,1,Yes,No,47,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,4871.05,Yes,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
8,5515-AKOAJ,Female,0,No,No,54,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.1,5440.9,Yes,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00
9,9094-AZPHK,Female,0,No,No,15,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.15,1415,No,telco_customer_churn.csv,2026-10-12 10:15:26.984771+00:00


## 3. Kiểm tra sau nạp (đối chiếu CSV ↔ Bronze)
So sánh trên cùng một bộ chỉ số: số dòng, số cột nghiệp vụ, số `customerID` duy nhất, số ô `TotalCharges` trống, tổng `MonthlyCharges`, phân bố `Churn`, và khớp từng `customerID`.

In [16]:
q = f'''
SELECT
  COUNT(*)                                              AS so_dong,
  COUNT(DISTINCT customerID)                            AS so_customer_id,
  COUNTIF(TRIM(TotalCharges) = '')                      AS total_charges_trong,
  ROUND(SUM(CAST(MonthlyCharges AS NUMERIC)), 2)        AS tong_monthly_charges,
  COUNTIF(Churn = 'Yes')                                AS churn_yes,
  COUNTIF(Churn = 'No')                                 AS churn_no
FROM `{BRONZE_TABLE_ID}`
'''
bq = client.query(q).to_dataframe().iloc[0]

csv = {
    "so_dong":              len(df_raw),
    "so_customer_id":       df_raw["customerID"].nunique(),
    "total_charges_trong":  (df_raw["TotalCharges"].str.strip() == "").sum(),
    "tong_monthly_charges": round(df_raw["MonthlyCharges"].astype(float).sum(), 2),
    "churn_yes":            (df_raw["Churn"] == "Yes").sum(),
    "churn_no":             (df_raw["Churn"] == "No").sum(),
}
n_cols_bq = len([f for f in table.schema if not f.name.startswith("_")])

check = pd.DataFrame({
    "Chỉ số": list(csv.keys()) + ["so_cot_nghiep_vu"],
    "CSV":    [float(v) for v in csv.values()] + [df_raw.shape[1]],
    "Bronze": [float(bq[k]) for k in csv] + [n_cols_bq],
})
check["Khớp"] = (check["CSV"].round(2) == check["Bronze"].round(2)).map({True: "Khớp", False: "Lệch"})
check

,Chỉ số,CSV,Bronze,Khớp
0,so_dong,7043.0,7043.0,Khớp
1,so_customer_id,7043.0,7043.0,Khớp
2,total_charges_trong,11.0,11.0,Khớp
3,tong_monthly_charges,456116.6,456116.6,Khớp
4,churn_yes,1869.0,1869.0,Khớp
5,churn_no,5174.0,5174.0,Khớp
6,so_cot_nghiep_vu,21.0,21.0,Khớp


In [17]:
# Khớp từng customerID giữa CSV và Bronze
ids_bq = set(client.query(f"SELECT customerID FROM `{BRONZE_TABLE_ID}`").to_dataframe()["customerID"])
ids_csv = set(df_raw["customerID"])
print("Có trong CSV nhưng thiếu ở Bronze:", len(ids_csv - ids_bq))
print("Có ở Bronze nhưng không có trong CSV:", len(ids_bq - ids_csv))

# Không có giá trị NULL nào được tạo ra trong quá trình nạp (dữ liệu gốc không có NULL)
null_expr = " + ".join([f"COUNTIF({c} IS NULL)" for c in df_raw.columns])
print("Số ô NULL trên 21 cột ở Bronze:",
      list(client.query(f"SELECT {null_expr} AS n FROM `{BRONZE_TABLE_ID}`").result())[0].n)

Có trong CSV nhưng thiếu ở Bronze: 0
Có ở Bronze nhưng không có trong CSV: 0
Số ô NULL trên 21 cột ở Bronze: 0


## 4. Chuẩn bị môi trường dbt
Cài `dbt-bigquery`, tạo project `dbt_telco`, cấu hình `profiles.yml` dùng chính phiên đăng nhập Google ở bước 0.2 (`method: oauth`) – **không cần file service-account key**.

Lưu ý cho phần dbt:
- `dataset: silver_telco` trong profile → model sẽ được tạo thẳng trong `silver_telco`. **Không** đặt thêm `+schema` trong `dbt_project.yml`, nếu không dbt sẽ tạo dataset mới tên `silver_telco_<schema>`.
- Nguồn Bronze được khai báo sẵn trong `models/sources.yml` → trong model gọi bằng `{{ source('bronze_telco', 'customers_raw') }}`.

In [18]:
!pip -q install dbt-bigquery
!dbt --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.2/118.2 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 M

In [19]:
import os, textwrap
DBT_DIR = "/content/dbt_telco"
os.makedirs(f"{DBT_DIR}/models", exist_ok=True)
os.makedirs("/root/.dbt", exist_ok=True)

open("/root/.dbt/profiles.yml", "w").write(textwrap.dedent(f'''
dbt_telco:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: {PROJECT_ID}
      dataset: {SILVER_DATASET}
      location: {LOCATION}
      threads: 4
      job_execution_timeout_seconds: 300
'''))

open(f"{DBT_DIR}/dbt_project.yml", "w").write(textwrap.dedent('''
name: dbt_telco
version: "1.0.0"
config-version: 2
profile: dbt_telco
model-paths: ["models"]
models:
  dbt_telco:
    +materialized: table
'''))

open(f"{DBT_DIR}/models/sources.yml", "w").write(textwrap.dedent(f'''
version: 2
sources:
  - name: {BRONZE_DATASET}
    database: {PROJECT_ID}
    schema: {BRONZE_DATASET}
    tables:
      - name: {BRONZE_TABLE}
        description: "Dữ liệu Telco Customer Churn gốc (21 cột STRING + _source_file, _loaded_at)"
'''))
print("✅ Đã tạo profiles.yml, dbt_project.yml, models/sources.yml")

✅ Đã tạo profiles.yml, dbt_project.yml, models/sources.yml


### 4.1. Kiểm tra kết nối dbt ↔ BigQuery



In [20]:
%cd /content/dbt_telco
!dbt debug

/content/dbt_telco
10:22:00  Running with dbt=1.12.5
10:22:00  dbt version: 1.12.5
10:22:00  python version: 3.13.16
10:22:00  python path: /usr/bin/python3
10:22:00  os info: Linux-6.6.122+-x86_64-with-glibc2.39
10:22:03  Using profiles dir at /root/.dbt
10:22:03  Using profiles.yml file at /root/.dbt/profiles.yml
10:22:03  Using dbt_project.yml file at /content/dbt_telco/dbt_project.yml
10:22:03  adapter type: bigquery
10:22:03  adapter version: 1.12.1
10:22:03  Configuration:
10:22:03    profiles.yml file [OK found and valid]
10:22:03    dbt_project.yml file [OK found and valid]
10:22:03  Required dependencies:
10:22:03   - git [OK found]

10:22:03  Connection:
10:22:03    method: oauth
10:22:03    database: avid-booster-511409-t2
10:22:03    execution_project: avid-booster-511409-t2
10:22:03    schema: silver_telco
10:22:03    location: asia-southeast1
10:22:03    priority: None
10:22:03    maximum_bytes_billed: None
10:22:03    reservation: None
10:22:03    impersonate_service_acc

---
## 5. Làm sạch bằng dbt – model staging & Silver *(Duy Lê)*
Viết `models/stg_telco_customers.sql` và `models/customers_cleaned.sql` (có thể tạo bằng `%%writefile`), sau đó chạy `!dbt run`.

## 6. Kiểm thử & đánh giá *(Khôi)*
Viết `models/schema.yml`, chạy `!dbt test`, so sánh `bronze_telco.customers_raw` với `silver_telco.customers_cleaned`.